In [1]:
!pip install -q -U docling langchain langchain-community langchain-huggingface chromadb
!pip install -q -U transformers accelerate bitsandbytes sentence-transformers gradio plotly pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 kB 6.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 684.0/684.0 kB 47.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.6/139.6 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 95.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 82.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 127.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 69.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 95.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0

In [ ]:
import os
import torch
import json
import re
import pandas as pd
import plotly.express as px
import gradio as gr
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, BitsAndBytesConfig
from langchain_huggingface import HuggingFacePipeline, HuggingFaceEmbeddings
from docling.document_converter import DocumentConverter
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document

print("1/3: Loading Embedding Model...")
embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
    model_kwargs={'device': 'cuda'}
)

print("2/3: Loading 4-bit LLM (Qwen2.5-7B)...")
model_id = "Qwen/Qwen2.5-7B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)

# Your updated 4-bit config for optimal T4 VRAM usage
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    quantization_config=bnb_config,
    torch_dtype=torch.float16
)

text_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=1024,
    temperature=0.1,
    return_full_text=False
)
llm = HuggingFacePipeline(pipeline=text_pipeline)

print("3/3: Initializing Document Parser & Database...")
converter = DocumentConverter()
vector_store = Chroma(embedding_function=embedding_model, persist_directory="./chroma_db")
uploaded_files_state = []

# ==========================================
# HELPER FUNCTIONS & JSON PARSER
# ==========================================

def extract_valid_json(text):
    """Robustly extracts the first valid JSON object from LLM text output."""
    # 1. Try extracting from Markdown code blocks ```json ... ```
    code_block_match = re.search(r"```(?:json)?\s*(\{.*?\})\s*```", text, re.DOTALL)
    if code_block_match:
        try:
            return json.loads(code_block_match.group(1))
        except json.JSONDecodeError:
            pass

    # 2. Use raw_decode to parse the first valid JSON object, ignoring trailing text
    start_idx = text.find('{')
    if start_idx != -1:
        try:
            decoder = json.JSONDecoder()
            obj, _ = decoder.raw_decode(text[start_idx:])
            return obj
        except json.JSONDecodeError:
            pass

    return None

# ==========================================
# CORE BACKEND FUNCTIONS
# ==========================================

def process_uploaded_files(file_paths):
    all_chunks = []
    for path in file_paths:
        print(f"Parsing {path} with Docling...")
        result = converter.convert(path)
        markdown_text = result.document.export_to_markdown()

        text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=1000,
            chunk_overlap=200,
            separators=["\n## ", "\n### ", "\n\n", "\n", " ", ""]
        )

        chunks = text_splitter.split_text(markdown_text)
        for i, chunk in enumerate(chunks):
            doc = Document(
                page_content=chunk,
                metadata={"source": os.path.basename(path), "chunk_id": i}
            )
            all_chunks.append(doc)

    vector_store.add_documents(all_chunks)
    return f"✅ Successfully ingested {len(file_paths)} documents and created {len(all_chunks)} vector chunks."

def chat_with_documents(query, selected_docs):
    retriever = vector_store.as_retriever(
        search_kwargs={"k": 5, "filter": {"source": {"$in": selected_docs}}}
    )
    docs = retriever.invoke(query)
    context = "\n\n".join([d.page_content for d in docs])

    prompt = f"""Use the following context to answer the query. If it's not in the context, say you don't know.
Context: {context}
Query: {query}
Answer:"""
    return llm.invoke(prompt)

def compare_documents(doc_a, doc_b, query="Highlight the key differences"):
    docs_a = vector_store.as_retriever(search_kwargs={"k": 3, "filter": {"source": doc_a}}).invoke(query)
    context_a = "\n".join([d.page_content for d in docs_a])

    docs_b = vector_store.as_retriever(search_kwargs={"k": 3, "filter": {"source": doc_b}}).invoke(query)
    context_b = "\n".join([d.page_content for d in docs_b])

    prompt = f"""Compare the two documents based on the query.
<Context_A source="{doc_a}">\n{context_a}\n</Context_A>
<Context_B source="{doc_b}">\n{context_b}\n</Context_B>
Query: {query}
Comparative Analysis:"""
    return llm.invoke(prompt)

def extract_key_points(selected_docs):
    docs = vector_store.as_retriever(search_kwargs={"k": 8, "filter": {"source": {"$in": selected_docs}}}).invoke("Summary")
    context = "\n".join([d.page_content for d in docs])

    prompt = f"""Extract the top 5 key points from the following context.
Format your output STRICTLY as a Markdown bulleted list using the '-' character.
Context: {context}
Key Points:"""
    return llm.invoke(prompt)

def generate_chart_from_docs(query, selected_docs):
    retriever = vector_store.as_retriever(
        search_kwargs={"k": 5, "filter": {"source": {"$in": selected_docs}}}
    )
    docs = retriever.invoke(query)
    context = "\n\n".join([d.page_content for d in docs])

    chart_prompt = f"""You are a data visualization assistant.
Extract tabular or quantitative data from the context to fulfill the query: "{query}".

Output MUST be a single valid JSON object with NO additional commentary or markdown code blocks.
Schema:
{{
  "title": "Chart Title",
  "chart_type": "bar",
  "x_label": "X-Axis Label",
  "y_label": "Y-Axis Label",
  "data": [
    {{"label": "Category A", "value": 100}},
    {{"label": "Category B", "value": 200}}
  ]
}}

Context:
{context}

JSON Output:"""

    raw_response = llm.invoke(chart_prompt)

    # Use the robust extractor
    chart_data = extract_valid_json(raw_response)

    if not chart_data or "data" not in chart_data:
        return None, f"Failed to extract valid chart JSON. Raw output received:\n{raw_response}"

    try:
        df = pd.DataFrame(chart_data["data"])
        chart_type = chart_data.get("chart_type", "bar").lower()
        title = chart_data.get("title", "Generated Visualization")
        x_col = "label"
        y_col = "value"

        if chart_type == "line":
            fig = px.line(df, x=x_col, y=y_col, title=title, markers=True)
        elif chart_type == "pie":
            fig = px.pie(df, names=x_col, values=y_col, title=title)
        elif chart_type == "scatter":
            fig = px.scatter(df, x=x_col, y=y_col, title=title)
        else:
            fig = px.bar(df, x=x_col, y=y_col, title=title, text_auto=True)

        fig.update_layout(
            template="plotly_dark",
            margin=dict(l=20, r=20, t=50, b=20),
            xaxis_title=chart_data.get("x_label", x_col),
            yaxis_title=chart_data.get("y_label", y_col)
        )
        return fig, f"✅ Successfully generated **{chart_type.upper()}** chart."

    except Exception as e:
        return None, f"Error rendering Plotly chart: {str(e)}"

# ==========================================
# GRADIO UI WRAPPERS
# ==========================================

def ui_upload(files):
    if not files: return "No files uploaded.", gr.update(), gr.update(), gr.update()
    paths = [f.name for f in files]
    global uploaded_files_state
    uploaded_files_state = [os.path.basename(p) for p in paths]
    status = process_uploaded_files(paths)
    return status, gr.update(choices=uploaded_files_state), gr.update(choices=uploaded_files_state), gr.update(choices=uploaded_files_state)

def ui_chat(query, selected_docs):
    if not selected_docs: return "Please select at least one document."
    return chat_with_documents(query, selected_docs)

def ui_compare(doc1, doc2):
    if not doc1 or not doc2: return "Please select two documents."
    return compare_documents(doc1, doc2)

def ui_summarize(selected_docs):
    if not selected_docs: return "Select at least one document."
    return extract_key_points(selected_docs)

def ui_generate_chart(query, selected_docs):
    if not selected_docs: return None, "Please select at least one document source."
    if not query: return None, "Please enter a chart description."
    return generate_chart_from_docs(query, selected_docs)

# ==========================================
# BUILD GRADIO LAYOUT
# ==========================================

with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🧠 Smart Document Intelligence Platform (Colab T4)")

    with gr.Row():
        # LEFT COLUMN: Ingestion
        with gr.Column(scale=1):
            file_input = gr.File(file_count="multiple", label="Upload PDF, DOCX, XLSX")
            upload_btn = gr.Button("Ingest Documents", variant="primary")
            status_text = gr.Textbox(label="System Status", interactive=False)
            doc_selector = gr.Dropdown(multiselect=True, label="Target Documents for Analysis")

        # RIGHT COLUMN: Workspaces
        with gr.Column(scale=2):
            with gr.Tab("💬 Multi-Doc Chat"):
                chat_query = gr.Textbox(label="Ask your documents...")
                chat_btn = gr.Button("Send")
                chat_output = gr.Markdown()

            with gr.Tab("📊 Visual Chart Generation"):
                chart_query = gr.Textbox(
                    label="Chart Request",
                    placeholder="e.g., Generate a bar chart showing Q3 revenue by region"
                )
                chart_btn = gr.Button("Generate Chart", variant="primary")
                chart_status = gr.Markdown()
                chart_plot = gr.Plot(label="Interactive Visualization")

            with gr.Tab("⚖️ Compare Documents"):
                with gr.Row():
                    comp_doc1 = gr.Dropdown(label="Document A")
                    comp_doc2 = gr.Dropdown(label="Document B")
                comp_btn = gr.Button("Find Differences")
                comp_output = gr.Markdown()

            with gr.Tab("📝 Auto Summary"):
                sum_btn = gr.Button("Extract Key Points")
                sum_output = gr.Markdown()

    # WIRE EVENTS
    upload_btn.click(
        ui_upload,
        inputs=[file_input],
        outputs=[status_text, doc_selector, comp_doc1, comp_doc2]
    )
    chat_btn.click(ui_chat, inputs=[chat_query, doc_selector], outputs=[chat_output])
    chart_btn.click(ui_generate_chart, inputs=[chart_query, doc_selector], outputs=[chart_plot, chart_status])
    comp_btn.click(ui_compare, inputs=[comp_doc1, comp_doc2], outputs=[comp_output])
    sum_btn.click(ui_summarize, inputs=[doc_selector], outputs=[sum_output])

# Launch the app
print("Starting UI...")
demo.launch(share=True, debug=True)

1/3: Loading Embedding Model...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

2/3: Loading 4-bit LLM (Qwen2.5-7B)...


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

3/3: Initializing Document Parser & Database...


...

Parsing /tmp/gradio/ef7fa9038f4c23bc67c087eda443db7c84134c11c3f4fa9a92a864e6775a3668/3M_2015_10K.pdf with Docling...


[INFO] 2026-07-26 12:13:49,732 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-07-26 12:13:49,798 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/PP-OCRv6_rec_small.onnx
[INFO] 2026-07-26 12:13:49,799 [RapidOCR] main.py:63: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/PP-OCRv6_rec_small.onnx


Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

Parsing /tmp/gradio/643e9743924f1956e8a261ff2f2bf15f27b5978f0e9ca6f83e7236799316d001/3M_2023Q2_10Q.pdf with Docling...


[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
